# Heads that think in loops: `head` × `head_loops` on typed-decisions

sysone 0.6 adds two settings to the decision head:

- `head` is **what the head reads**: `"compact"` (one vector per option and a summary vector, the
  head sysone has always used) or `"full"` (those plus every token's vector, so it can look back
  at the text);
- `head_loops` is **how many times the head runs**, with the same weights: more computation per
  decision, not more parameters.

This notebook measures whether either helps, on
[typed-decisions](https://huggingface.co/datasets/LocalLLaMA/typed-decisions), in two steps:

| step | what | runs | expected time, Colab L4 |
|---|---|---|---|
| 1 | Ettin-encoder-150m: `compact`/`full` × 1/3 loops, 3 seeds each | 12 | 20–30 min each, about 5 hours |
| 2 | Ettin-encoder-400m: the current head and step 1's winner, 3 seeds each | 6 | about 1 hour each |

The 2×2 separates the two effects: `compact` with 3 loops is iterating alone, `full` with 1 loop
is reading the text alone, and `full` with 3 loops is both. Every 3-loop model is also scored with
1 to 6 loops after training: if accuracy rises with loops, the loops are doing work.

> **⚠️ Use a GPU runtime** (Runtime → Change runtime type → L4 or A100) and connect Google Drive
> when asked. Every run autosaves to Drive and writes its results there when it finishes. If the
> runtime disconnects, run the notebook again from the top: finished runs are read back, and an
> unfinished one continues from its last save. You can stop after any run; the summaries use
> whatever has finished.

In [ ]:
# sysone 0.6 isn't on PyPI yet: this installs it from its development branch. Restart the
# runtime first if an older sysone is installed in this session.
import importlib.util

SYSONE = "sysone @ git+https://github.com/ForwardMetric/sysone.git@milestone-6-head-loops"
if importlib.util.find_spec("sysone") is None:
    %pip install -q "$SYSONE"
if importlib.util.find_spec("datasets") is None:
    %pip install -q datasets

In [ ]:
import json
import math
import statistics
import time
from pathlib import Path

import pandas as pd
import torch
from datasets import load_dataset

import sysone
from sysone import EncoderModel

assert "head_loops" in EncoderModel().get_params(), "this needs sysone 0.6: restart the runtime and run again"
print("sysone", sysone.__version__)

# Results and checkpoints go to Google Drive on Colab, so a disconnect loses nothing; elsewhere
# they go to a local folder.
try:
    from google.colab import drive

    drive.mount("/content/drive")
    OUT = Path("/content/drive/MyDrive/sysone-head-loops")
except ImportError:
    OUT = Path("sysone-head-loops")
OUT.mkdir(parents=True, exist_ok=True)
print("saving to", OUT)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (this will be very slow)")

## Load the data

All four workflows: 1,200 training and 400 test records, five questions each, with soft labels
(each answer is the raters' probability spread).

In [ ]:
def load_rows(split):
    ds = load_dataset("LocalLLaMA/typed-decisions", "all", split=split)
    return [{"state": r["state"], "questions": r["questions"], "gold": r["gold"], "category": r["workflow"]}
            for r in ds]


train = load_rows("train")
test = load_rows("test")
print("%d training records, %d test records" % (len(train), len(test)))

## Scoring, question by question

The leaderboard's formulas (from the dataset's `bench/score_with_jobs.py`): accuracy is the
predicted top option matching the raters' top option (a tie goes to the first option), plus KL
from the raters' distribution, Brier summed over options and ECE in 10 bins.

Every test question is also kept with its type and the **raters' margin**, the gap between their
top two answers, so results can be broken down: a gain on questions the raters clearly agree on
means something different from one on near-ties.

In [ ]:
QUESTION_TYPES = {}  # (record index, question id) -> type
for i, r in enumerate(test):
    for qid, q in json.loads(r["questions"]).items():
        QUESTION_TYPES[(i, qid)] = q["type"]


def score_questions(model, rows):
    """One record per test question: type, raters' margin, hit, KL, Brier and confidence."""
    results = model.predict([{"state": r["state"], "questions": r["questions"]} for r in rows])
    out = []
    for i, (row, result) in enumerate(zip(rows, results)):
        for qid, gold in json.loads(row["gold"]).items():
            answer = result["answers"][qid]
            probs = answer.get("probabilities") or {"false": 1 - answer["noul"], "true": answer["noul"]}
            options = list(gold["probabilities"])
            g = [float(gold["probabilities"][o]) for o in options]
            p = [float(probs.get(o, 0.0)) for o in options]
            total = sum(p)
            p = [v / total for v in p] if total > 0 else [1 / len(p)] * len(p)
            top = sorted(g, reverse=True)
            out.append({
                "record": i, "question": qid, "type": QUESTION_TYPES[(i, qid)],
                "margin": top[0] - (top[1] if len(top) > 1 else 0.0),
                "hit": p.index(max(p)) == g.index(max(g)),
                "kl": sum(gi * math.log(gi / max(pi, 1e-9)) for gi, pi in zip(g, p) if gi > 0),
                "brier": sum((pi - gi) ** 2 for pi, gi in zip(p, g)),
                "confidence": max(p),
            })
    return out


def leaderboard(questions):
    n = len(questions)
    bins = [[0, 0, 0.0] for _ in range(10)]  # count, hits, confidence sum
    for q in questions:
        b = bins[min(int(q["confidence"] * 10), 9)]
        b[0] += 1
        b[1] += q["hit"]
        b[2] += q["confidence"]
    return {"accuracy": sum(q["hit"] for q in questions) / n,
            "kl_from_gold": sum(q["kl"] for q in questions) / n,
            "brier": sum(q["brier"] for q in questions) / n,
            "ece": sum(t / n * abs(h / t - c / t) for t, h, c in bins if t)}

## Train, score and save, resumably

`run` trains one model and writes to `OUT/<name>/`: autosaves in `checkpoint/`, and
`results.json` with the leaderboard scores, every test question's result, the training time and,
for models with more than one loop, the accuracy when predicting with 1 to 6 loops. The model
itself isn't kept (18 runs would fill a Drive); add `model.save(...)` if you want one.

In [ ]:
CURVE = range(1, 7)


def run(name, model_name, **params):
    folder = OUT / name
    results_file = folder / "results.json"
    if results_file.exists():
        print("%s: already done" % name)
        return json.loads(results_file.read_text())

    checkpoint = folder / "checkpoint"
    start = time.perf_counter()
    if (checkpoint / "run.json").exists():
        info = json.loads((checkpoint / "run.json").read_text())
        print("%s: resuming from step %d (%s)" % (name, info["step"], info["reason"]))
        model = EncoderModel.load(checkpoint)
        if info["reason"] != "finished":
            model.fit(train)
    else:
        print("%s: training" % name)
        model = EncoderModel(model_name, checkpoint_dir=str(checkpoint), **params)
        model.fit(train)
    fit_minutes = (time.perf_counter() - start) / 60

    questions = score_questions(model, test)
    results = {"name": name, "model_name": model_name, "params": params, "fit_minutes": fit_minutes,
               "leaderboard": leaderboard(questions), "questions": questions}
    loops = model.head_loops
    if loops > 1:
        results["curve"] = {}
        for n in CURVE:
            model.set_params(head_loops=n)
            results["curve"][str(n)] = leaderboard(score_questions(model, test))["accuracy"]
        model.set_params(head_loops=loops)
    results_file.write_text(json.dumps(results, default=str))
    print("%s: leaderboard accuracy %.4f, %.1f minutes" % (name, results["leaderboard"]["accuracy"], fit_minutes))
    return results


# Settings shared by every run. Validation runs about ten times per epoch; training stops once
# validation loss hasn't improved for 3 epochs and keeps the best weights.
COMMON = dict(tuning="full", epochs=10, batch_size=32, checkpoint_every=300)
HEADS = [("compact", 1), ("compact", 3), ("full", 1), ("full", 3)]
SEEDS = [0, 1, 2]


def label(head, loops):
    return "%s × %d" % (head, loops)

## Step 1: the 2×2 on Ettin-encoder-150m

22 layers, 768 wide, like ModernBERT-base, so it uses ModernBERT-base's measured learning rate
(5e-5). Runs go seed by seed, so after the first four you already have one full 2×2 to look at.

In [ ]:
step1 = {}
for seed in SEEDS:
    for head, loops in HEADS:
        name = "ettin-150m-%s-%d-seed%d" % (head, loops, seed)
        step1[(head, loops, seed)] = run(name, "jhu-clsp/ettin-encoder-150m", backbone_lr=5e-5, head=head,
                                         head_loops=loops, seed=seed, **COMMON)

### Results

Mean and spread across seeds, and the **paired difference**: each configuration minus
`compact × 1` on the same seed, which removes most of the seed-to-seed noise. A gain that is
smaller than its spread across seeds isn't one yet.

In [ ]:
def summarize(results, baseline=("compact", 1)):
    rows = {}
    for head, loops in sorted({(h, l) for h, l, _ in results}):
        seeds = [s for h, l, s in results if (h, l) == (head, loops)]
        acc = [results[(head, loops, s)]["leaderboard"]["accuracy"] for s in seeds]
        diffs = [results[(head, loops, s)]["leaderboard"]["accuracy"] - results[baseline + (s,)]["leaderboard"]["accuracy"]
                 for s in seeds if baseline + (s,) in results]
        rows[label(head, loops)] = {
            "seeds": len(seeds),
            "accuracy": statistics.mean(acc),
            "accuracy sd": statistics.stdev(acc) if len(acc) > 1 else float("nan"),
            "vs compact × 1": statistics.mean(diffs) if diffs else float("nan"),
            "diff sd": statistics.stdev(diffs) if len(diffs) > 1 else float("nan"),
            **{k: statistics.mean(results[(head, loops, s)]["leaderboard"][k] for s in seeds)
               for k in ("kl_from_gold", "brier", "ece")},
            "minutes": statistics.mean(results[(head, loops, s)]["fit_minutes"] for s in seeds),
        }
    return pd.DataFrame.from_dict(rows, orient="index")


summarize(step1).round(4)

### Does accuracy rise with loops?

Each 3-loop model predicting with 1 to 6 loops (it trained with 3), averaged over seeds. Rising
up to 3 means the loops do work; flat means any gain comes from elsewhere (for `full`, reading the
text). Beyond 3 the model is running loops it never trained with.

In [ ]:
def curves(results):
    rows = {}
    for head, loops in sorted({(h, l) for h, l, _ in results if l > 1}):
        runs_ = [results[(h, l, s)] for h, l, s in results if (h, l) == (head, loops)]
        rows[label(head, loops)] = {n: statistics.mean(r["curve"][str(n)] for r in runs_) for n in CURVE}
    return pd.DataFrame.from_dict(rows, orient="index")


curves(step1).round(4)

### Where the gains come from

Accuracy by the raters' margin (near-ties, where even raters split, versus clear questions) and by
question type, averaged over seeds.

In [ ]:
BUCKETS = [(0.0, 0.1, "near-tie (≤0.1)"), (0.1, 0.3, "middle (0.1–0.3)"), (0.3, 1.01, "clear (>0.3)")]


def breakdown(results, by):
    rows = {}
    for head, loops in sorted({(h, l) for h, l, _ in results}):
        qs = [q for h, l, s in results if (h, l) == (head, loops) for q in results[(h, l, s)]["questions"]]
        if by == "margin":
            groups = {name: [q for q in qs if lo <= q["margin"] < hi] for lo, hi, name in BUCKETS}
        else:
            groups = {}
            for q in qs:
                groups.setdefault(q["type"], []).append(q)
        rows[label(head, loops)] = {name: sum(q["hit"] for q in g) / len(g) for name, g in groups.items() if g}
    return pd.DataFrame.from_dict(rows, orient="index")


display(breakdown(step1, "margin").round(4))
display(breakdown(step1, "type").round(4))

## Step 2: confirm on Ettin-encoder-400m

Set `WINNER` to step 1's best configuration. This trains it and the current head (`compact × 1`)
with 3 seeds each, at the learning rate of the earlier 0.8025 run (3e-5).

Only adopt the winner if it gains about a point or more over `compact × 1` on average across the
seeds, with a spread smaller than the gain, and (for 3 loops) a curve that rises.

In [ ]:
WINNER = ("full", 3)  # step 1's best (head, loops)

step2 = {}
for seed in SEEDS:
    for head, loops in [("compact", 1), WINNER]:
        name = "ettin-400m-%s-%d-seed%d" % (head, loops, seed)
        step2[(head, loops, seed)] = run(name, "jhu-clsp/ettin-encoder-400m", backbone_lr=3e-5, head=head,
                                         head_loops=loops, seed=seed, **COMMON)

display(summarize(step2).round(4))
if WINNER[1] > 1:
    display(curves(step2).round(4))
display(breakdown(step2, "margin").round(4))

## Reading the results

- Ettin-encoder-400m's earlier best was 0.8025 (one seed, `compact × 1`); the best entries on the
  leaderboard's fine-tuned table are 4B models at about 0.82.
- About 18% of test questions have the raters' top two answers within 10 points, so no model will
  get all of those right; gains there are worth less than gains on the clear questions.
- KL and Brier show whether the probabilities, not just the top answer, got closer to the raters'.